In [1]:
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown

def plot_results_dropdown(csv_file='Ames_Regression_interaktiv_log.csv',
                          last_n=20, run_range=None, method_filter=None,
                          startRunNo=None):
    """
    Erstellt einen interaktiven Scatterplot (R² vs. RMSE) mit Filtern nach
    Run‑Bereich, Methode, Kategorie (Präfix aus Kommentar) und Dateiname.
    Zwei Run‑Dropdowns erlauben den Vergleich zweier Läufe in einer Tabelle.

    Parameter
    ----------
    csv_file : str, default='Ames_Regression_interaktiv_log.csv'
        Pfad zur Log‑CSV.
    last_n : int, default=20
        Anzahl der letzten Runs, die initial angezeigt werden (nur wenn run_range=None und startRunNo=None).
    run_range : tuple (min_run, max_run), optional
        Expliziter Run‑Bereich (überschreibt last_n und startRunNo).
    method_filter : str oder Liste von str, optional
        Auf diese Methode(n) vorfiltern (bestimmt die initial sichtbaren Methoden und die Auswahllisten).
    startRunNo : int, optional
        Minimaler Run für den initialen Bereich (max_run = maximaler Run im Datensatz).
    """

    # ====== 1. Globale Variablen ======
    current_df = None          # vollständiger Datensatz
    global_min_run = global_max_run = None
    all_methods_full = []
    all_prefixes_raw_full = []
    has_none_full = False
    category_options_full = []
    all_filenames_full = []

    # ====== 2. Hilfsfunktionen ======
    def parse_hyperparams(s):
        """Parst Hyperparameter-Strings der Form 'key=value, key2=(a,b), ...' unter Beachtung von Klammern."""
        if not s or pd.isna(s):
            return {}
        params = {}
        i = 0
        length = len(s)
        while i < length:
            # Leerzeichen überspringen
            while i < length and s[i].isspace():
                i += 1
            if i >= length:
                break
            # Schlüssel extrahieren
            start_key = i
            while i < length and s[i] != '=':
                i += 1
            key = s[start_key:i].strip()
            if i >= length or s[i] != '=':
                # Fehlerhafte Syntax: kein '=' gefunden, überspringe den Rest
                break
            i += 1  # '=' überspringen
            # Leerzeichen vor dem Wert überspringen
            while i < length and s[i].isspace():
                i += 1
            if i >= length:
                params[key] = ''
                break
            # Wert extrahieren – Klammern beachten
            if s[i] == '(':
                # Klammerausdruck – zähle Klammern
                depth = 1
                start_val = i
                i += 1
                while i < length and depth > 0:
                    if s[i] == '(':
                        depth += 1
                    elif s[i] == ')':
                        depth -= 1
                    i += 1
                value = s[start_val:i].strip()
            else:
                # Einfacher Wert bis zum nächsten Komma
                start_val = i
                while i < length and s[i] != ',':
                    i += 1
                value = s[start_val:i].strip()
            params[key] = value
            # Komma überspringen, falls vorhanden
            if i < length and s[i] == ',':
                i += 1
        return params

    def create_figure(df_filtered):
        """Erstellt die Plotly-Figur basierend auf dem übergebenen DataFrame."""
        df_filtered = df_filtered.copy()
        x_min = df_filtered['R_Quadrat'].min()
        x_max = df_filtered['R_Quadrat'].max()
        y_min = df_filtered['RMSE'].min()
        y_max = df_filtered['RMSE'].max()
        x_range = x_max - x_min
        y_range = y_max - y_min

        df_filtered['R_Quadrat_plot'] = df_filtered['R_Quadrat'].copy()
        df_filtered['RMSE_plot'] = df_filtered['RMSE'].copy()

        dup_groups = df_filtered.groupby(['R_Quadrat', 'RMSE'], group_keys=False)
        offset_applied = False
        for (x_orig, y_orig), group in dup_groups:
            count = len(group)
            if count > 1:
                offset_applied = True
                centered = [i - (count - 1) / 2 for i in range(count)]
                step = 0.005
                for idx_in_group, row_idx in enumerate(group.index):
                    offset_x = centered[idx_in_group] * step * x_range
                    offset_y = centered[idx_in_group] * step * y_range
                    df_filtered.at[row_idx, 'R_Quadrat_plot'] = x_orig + offset_x
                    df_filtered.at[row_idx, 'RMSE_plot'] = y_orig + offset_y

        symbol_list = [
            'circle', 'square', 'diamond', 'cross', 'x',
            'triangle-up', 'triangle-down', 'triangle-left', 'triangle-right',
            'pentagon', 'hexagon', 'octagon', 'star', 'hexagram',
            'star-triangle-up', 'star-triangle-down', 'star-square', 'star-diamond'
        ]
        # Symbole basierend auf Postfix
        unique_postfixes = df_filtered['symbol_postfix'].dropna().unique()
        if len(unique_postfixes) > 15:
            unique_postfixes = unique_postfixes[:15]
        postfix_to_symbol = {pref: sym for pref, sym in zip(unique_postfixes, symbol_list)}
        default_symbol = 'circle'

        df_filtered['RMSLE_str'] = df_filtered['RMSLE'].apply(
            lambda x: f"{x:.4f}" if pd.notna(x) else "N/A"
        )

        fig = go.Figure()

        y_plot_min = df_filtered['RMSE_plot'].min()
        y_plot_max = df_filtered['RMSE_plot'].max()
        y_range_plot = [y_plot_min - 0.2 * (y_plot_max - y_plot_min),
                        y_plot_max + 0.2 * (y_plot_max - y_plot_min)]

        for method in df_filtered['Methode'].unique():
            df_meth = df_filtered[df_filtered['Methode'] == method]
            symbols_for_meth = [
                postfix_to_symbol.get(p, default_symbol)
                for p in df_meth['symbol_postfix']
            ]
            fig.add_trace(go.Scatter(
                x=df_meth['R_Quadrat_plot'],
                y=df_meth['RMSE_plot'],
                mode='markers+text',
                text=df_meth['RunNo'],
                textposition="top center",
                marker=dict(size=10, symbol=symbols_for_meth),
                name=method,
                customdata=df_meth[['RunNo', 'Kommentar', 'Methode', 'Scaler', 'Test_Size',
                                     'RMSE', 'MAE', 'RMSLE_str', 'R_Quadrat','scriptname','Dateiname', 'AnzahlDaten']].values,
                hovertemplate=(
                    'Run: %{customdata[0]}<br>' +
                    'Comment: %{customdata[1]}<br>' +
                    'Method: %{customdata[2]}<br>' +
                    'Scaler: %{customdata[3]}<br>' +
                    'Test Size: %{customdata[4]}<br>' +
                    'RMSE: %{customdata[5]:.2f}<br>' +
                    'MAE: %{customdata[6]:.2f}<br>' +
                    'RMSLE: %{customdata[7]}<br>' +
                    'R²: %{customdata[8]:.4f}<br>' +
                    'Script: %{customdata[9]}<br>' + 
                    'Data: %{customdata[10]} (%{customdata[11]})<br>' + 
                    '<extra></extra>'
                )
            ))

        # Zusätzliche Legendeinträge für Postfixe (unsichtbare Punkte) – nur aus gefilterten Daten
        for postfix, symbol in postfix_to_symbol.items():
            fig.add_trace(go.Scatter(
                x=[None], y=[None],
                mode='markers',
                marker=dict(symbol=symbol, size=10, color='black'),
                name=f"☞ {postfix}",
                showlegend=True
            ))
        if df_filtered['symbol_postfix'].isna().any():
            fig.add_trace(go.Scatter(
                x=[None], y=[None],
                mode='markers',
                marker=dict(symbol=default_symbol, size=10, color='black'),
                name="ohne Postfix",
                showlegend=True
            ))

        fig.update_layout(
            title='Scatterplot R² vs RMSE (Run-Bereich, Methode, Kategorie, Datei wählbar)',
            xaxis_title='R²',
            yaxis_title='RMSE',
            yaxis_range=y_range_plot,
            legend=dict(yanchor="top", y=0.99, xanchor="left", x=1.05, traceorder="normal")
        )

        return fig, offset_applied

    def create_info_table(current_row, compare_row):
        """Erstellt die HTML-Tabelle mit den Details der beiden Runs."""

        # ---- Hilfsfunktionen für robuste Typkonvertierung und Vergleich ----
        def ensure_scalar(x):
            """Wandelt pandas- oder numpy-Objekte in Python-Skalare um."""
            if hasattr(x, 'item'):
                return x.item()
            return x

        def safe_neq(a, b):
            """Vergleicht zwei Werte sicher, auch wenn sie NaN oder pandas-Objekte sind."""
            a = ensure_scalar(a)
            b = ensure_scalar(b)
            # Behandle None
            if a is None or b is None:
                return a != b
            # Behandle Gleitkommazahlen mit Toleranz
            if isinstance(a, (int, float)) and isinstance(b, (int, float)):
                if pd.isna(a) and pd.isna(b):
                    return False
                if pd.isna(a) or pd.isna(b):
                    return True
                return abs(a - b) > 1e-9
            # Standardvergleich
            return a != b

        # Prüfen, ob ein Vergleich mit einem anderen Run vorliegt
        is_comparing = (compare_row is not None and compare_row.name != current_row.name)

        # ---- Werte aus current_row holen und in Skalare wandeln ----
        # Spalte 1
        comment = ensure_scalar(current_row['Kommentar'])
        filename = ensure_scalar(current_row['Dateiname'])
        method = ensure_scalar(current_row['Methode'])
        zeitpunkt = ensure_scalar(current_row['Zeitpunkt'])          # pandas Timestamp
        rechenzeit = ensure_scalar(current_row['Rechenzeit'])
        min_price = ensure_scalar(current_row['Min_Price'])
        max_price = ensure_scalar(current_row['Max_Price'])

        # Spalte 2
        mae = ensure_scalar(current_row['MAE'])
        rmse = ensure_scalar(current_row['RMSE'])
        rmsle = ensure_scalar(current_row['RMSLE'])
        r2 = ensure_scalar(current_row['R_Quadrat'])

        # Spalte 3
        use_log = ensure_scalar(current_row['Use_Log'])
        scale_target = ensure_scalar(current_row['Scale_target'])
        scaler = ensure_scalar(current_row['Scaler'])
        test_size = ensure_scalar(current_row['Test_Size'])
        random_state = ensure_scalar(current_row['random_state'])
        shuffle_split = ensure_scalar(current_row['Shuffle_Split'])
        stratify = ensure_scalar(current_row['Stratify'])
        n_bins = ensure_scalar(current_row['N_Bins'])

        # Spalte 4
        hyperparameter = ensure_scalar(current_row['Hyperparameter']) if pd.notna(current_row['Hyperparameter']) else ''
        variablen_str = ensure_scalar(current_row['Variablen'])

        scriptname = ensure_scalar(current_row['scriptname'])
        anzahl_daten = ensure_scalar(current_row['AnzahlDaten'])

        # ---- Vergleichsdaten (nur wenn is_comparing) ----
        if is_comparing:
            l_comment = ensure_scalar(compare_row['Kommentar'])
            l_filename = ensure_scalar(compare_row['Dateiname'])
            l_method = ensure_scalar(compare_row['Methode'])
            l_zeitpunkt = ensure_scalar(compare_row['Zeitpunkt']).strftime('%d.%m.%Y %H:%M:%S')
            l_rechenzeit = ensure_scalar(compare_row['Rechenzeit'])
            l_min_price = ensure_scalar(compare_row['Min_Price'])
            l_max_price = ensure_scalar(compare_row['Max_Price'])
            l_mae = ensure_scalar(compare_row['MAE'])
            l_rmse = ensure_scalar(compare_row['RMSE'])
            l_rmsle = ensure_scalar(compare_row['RMSLE'])
            l_r2 = ensure_scalar(compare_row['R_Quadrat'])
            l_use_log = ensure_scalar(compare_row['Use_Log'])
            l_scale_target = ensure_scalar(compare_row['Scale_target'])
            l_scaler = ensure_scalar(compare_row['Scaler'])
            l_test_size = ensure_scalar(compare_row['Test_Size'])
            l_random_state = ensure_scalar(compare_row['random_state'])
            l_shuffle_split = ensure_scalar(compare_row['Shuffle_Split'])
            l_stratify = ensure_scalar(compare_row['Stratify'])
            l_n_bins = ensure_scalar(compare_row['N_Bins'])
            l_hyperparameter = ensure_scalar(compare_row['Hyperparameter']) if pd.notna(compare_row['Hyperparameter']) else ''
            l_variablen_str = ensure_scalar(compare_row['Variablen'])
            l_scriptname = ensure_scalar(compare_row['scriptname'])
            l_anzahl_daten = ensure_scalar(compare_row['AnzahlDaten'])
            l_FilteredData = ensure_scalar(compare_row['FilteredData'])
            hinweis_text = (f"<b>Hinweis</b>: Fett = nur in Run {current_row['RunNo']} (aktuell), "
                            f"durchgestrichen = nur in Run {compare_row['RunNo']} (Vergleich).")
        else:
            # Kein Vergleich oder Selbstvergleich
            l_comment = l_filename = l_method = l_zeitpunkt = l_rechenzeit = None
            l_min_price = l_max_price = l_mae = l_rmse = l_rmsle = l_r2 = None
            l_use_log = l_scale_target = l_scaler = l_test_size = None
            l_random_state = l_shuffle_split = l_stratify = l_n_bins = None
            l_hyperparameter = l_variablen_str = None
            l_scriptname = None
            l_anzahl_daten = None
            l_FilteredData = None
            hinweis_text = "<b>Hinweis</b>: Vergleich mit sich selbst – keine Markierungen von Unterschieden."

        # ---- Hilfsfunktion für bedingte Fettdarstellung (nur bei is_comparing) ----
        def fmt_val(val, cond):
            if is_comparing and cond:
                return f"<b>{val}</b>"
            else:
                return str(val)

        # Hilfsfunktion für Werte mit Vergleich (aktuell fett bei Unterschied, Vergleich normal in Klammern)
        def format_with_compare(current_val, compare_val, fmt_str, is_diff):
            current_val = ensure_scalar(current_val)
            try:
                current_formatted = fmt_str.format(current_val)
            except:
                current_formatted = str(current_val)
            if is_comparing and compare_val is not None:
                compare_val = ensure_scalar(compare_val)
                try:
                    compare_formatted = fmt_str.format(compare_val)
                except:
                    compare_formatted = str(compare_val)
                if is_diff:
                    return f"<b>{current_formatted}</b> ({compare_formatted})"
                else:
                    return f"{current_formatted} ({compare_formatted})"
            else:
                return current_formatted

        indent = "&nbsp;" * 10

        # ---- Spalte 1 zusammenbauen ----
        comment_line = f"Comment: {fmt_val(comment, is_comparing and safe_neq(comment, l_comment))}<br>"
        filename_line = f"Filename: {fmt_val(filename, is_comparing and safe_neq(filename, l_filename))}<br>"
        filtered_data_current = ensure_scalar(current_row['FilteredData'])
        filtered_data_compare = l_FilteredData if is_comparing else None
        filtered_data_diff = is_comparing and safe_neq(filtered_data_current, filtered_data_compare)
        filtered_data_display = format_with_compare(filtered_data_current, filtered_data_compare, "{}", filtered_data_diff)
        daten_line = f"{indent}Features: {current_row['Anzahl_Features']}, Daten: {fmt_val(anzahl_daten, is_comparing and safe_neq(anzahl_daten, l_anzahl_daten))} , Gefiltert: {filtered_data_display}<br>"
        script_line = f"Script: {fmt_val(scriptname, is_comparing and safe_neq(scriptname, l_scriptname))}<br>"

        method_display = fmt_val(method, is_comparing and safe_neq(method, l_method))
        if is_comparing:
            method_display += f" ({l_method})"
        method_line = f"Method: {method_display}<br>"

        zeitpunkt_str = zeitpunkt.strftime('%d.%m.%Y %H:%M:%S')
        zeitpunkt_display = zeitpunkt_str
        if is_comparing:
            zeitpunkt_display += f" ({l_zeitpunkt})"
        zeitpunkt_line = f"Zeitpunkt: {zeitpunkt_display}<br>"

        rechenzeit_diff = is_comparing and safe_neq(rechenzeit, l_rechenzeit)
        rechenzeit_display = format_with_compare(rechenzeit, l_rechenzeit, "{:.2f}s", rechenzeit_diff)

        min_price_diff = is_comparing and safe_neq(min_price, l_min_price)
        min_price_display = format_with_compare(min_price, l_min_price, "{:.0f}", min_price_diff)

        max_price_diff = is_comparing and safe_neq(max_price, l_max_price)
        max_price_display = format_with_compare(max_price, l_max_price, "{:.0f}", max_price_diff)

        spalte1 = (
            comment_line +
            filename_line +
            daten_line +
            script_line +
            method_line +
            zeitpunkt_line +
            f"Rechenzeit: {rechenzeit_display}<br>" +
            f"Min Price: {min_price_display}<br>" +
            f"Max Price: {max_price_display}<br>"
        )

        # ---- Spalte 2 (Metriken) ----
        spalte2_lines = []

        # MAE
        mae_diff = is_comparing and safe_neq(mae, l_mae)
        mae_line = "MAE: "
        if mae_diff:
            mae_line += f"<b>{mae:.2f}</b>"
        else:
            mae_line += f"{mae:.2f}"
        if is_comparing and l_mae is not None and l_mae != 0:
            rel = (mae - l_mae) / l_mae * 100
            mae_line += f" &nbsp;&nbsp;({rel:+.1f}%)"
        spalte2_lines.append(mae_line)

        # RMSE
        rmse_diff = is_comparing and safe_neq(rmse, l_rmse)
        rmse_line = "RMSE: "
        if rmse_diff:
            rmse_line += f"<b>{rmse:.2f}</b>"
        else:
            rmse_line += f"{rmse:.2f}"
        if is_comparing and l_rmse is not None and l_rmse != 0:
            rel = (rmse - l_rmse) / l_rmse * 100
            rmse_line += f" &nbsp;&nbsp;({rel:+.1f}%)"
        spalte2_lines.append(rmse_line)

        # RMSLE
        rmsle_diff = is_comparing and safe_neq(rmsle, l_rmsle)
        rmsle_str = f"{rmsle:.4f}" if pd.notna(rmsle) else "N/A"
        rmsle_line = "RMSLE: "
        if rmsle_diff:
            rmsle_line += f"<b>{rmsle_str}</b>"
        else:
            rmsle_line += rmsle_str
        if is_comparing and pd.notna(rmsle) and pd.notna(l_rmsle) and l_rmsle != 0:
            rel = (rmsle - l_rmsle) / l_rmsle * 100
            rmsle_line += f" &nbsp;&nbsp;({rel:+.1f}%)"
        spalte2_lines.append(rmsle_line)

        # R²
        r2_diff = is_comparing and safe_neq(r2, l_r2)
        r2_line = "R²: "
        if r2_diff:
            r2_line += f"<b>{r2:.4f}</b>"
        else:
            r2_line += f"{r2:.4f}"
        if is_comparing and l_r2 is not None and l_r2 != 0:
            rel = (r2 - l_r2) / l_r2 * 100
            r2_line += f" &nbsp;&nbsp;({rel:+.1f}%)"
        spalte2_lines.append(r2_line)

        spalte2 = "<br>".join(spalte2_lines)

        # ---- Spalte 3 (Einstellungen) ----
        use_log_cond = is_comparing and safe_neq(use_log, l_use_log)
        scale_target_cond = is_comparing and safe_neq(scale_target, l_scale_target)
        scaler_cond = is_comparing and safe_neq(scaler, l_scaler)
        test_size_cond = is_comparing and safe_neq(test_size, l_test_size)
        random_state_cond = is_comparing and safe_neq(random_state, l_random_state)
        shuffle_split_cond = is_comparing and safe_neq(shuffle_split, l_shuffle_split)
        stratify_cond = is_comparing and safe_neq(stratify, l_stratify)
        n_bins_cond = is_comparing and safe_neq(n_bins, l_n_bins)
        combined_cond = stratify_cond or n_bins_cond

        spalte3 = f"""
        Zielvariable logarithmiert: {fmt_val(use_log, use_log_cond)}<br>
        Zielvariable skaliert: {fmt_val(scale_target, scale_target_cond)}<br>
        Scaler: {fmt_val(scaler, scaler_cond)}<br>
        test_size: {fmt_val(test_size, test_size_cond)}<br>
        random_state: {fmt_val(random_state, random_state_cond)}<br>
        shuffle_split: {fmt_val(shuffle_split, shuffle_split_cond)}<br>
        {fmt_val(f'stratify: {stratify}, n_bins: {n_bins}', combined_cond)}<br>
        """

        # ---- Spalte 4 (Hyperparameter) ----
        current_hp = parse_hyperparams(hyperparameter)
        if is_comparing:
            l_hp = parse_hyperparams(l_hyperparameter)
        else:
            l_hp = {}

        hp_lines = []
        same_method = (is_comparing and not safe_neq(method, l_method))

        if current_hp:
            for key in sorted(current_hp.keys(), reverse=True):
                val = current_hp[key]
                cond = same_method and (key not in l_hp or safe_neq(l_hp.get(key), val))
                hp_lines.append(f"{key} = {fmt_val(val, cond)}")
        else:
            hp_lines.append("—")
        spalte4 = "<br>".join(hp_lines)

        # ---- Variablen ----
        vars_current = [v.strip() for v in variablen_str.split(',')] if pd.notna(variablen_str) else []
        if is_comparing and l_variablen_str is not None:
            vars_last = [v.strip() for v in l_variablen_str.split(',')]
        else:
            vars_last = []

        all_vars_set = set(vars_current) | set(vars_last)
        all_vars_sorted = sorted(all_vars_set, reverse=True)

        def format_var(var):
            in_current = var in vars_current
            in_last = var in vars_last
            if is_comparing:
                if in_current and in_last:
                    return var
                elif in_current and not in_last:
                    return f"<b>{var}</b>"
                elif not in_current and in_last:
                    return f"<s>{var}</s>"
                else:
                    return var
            else:
                # Bei Selbstvergleich keine Hervorhebungen
                return var

        vars_lines = []
        for i in range(0, len(all_vars_sorted), 10):
            block = all_vars_sorted[i:i+10]
            formatted_block = [format_var(var) for var in block]
            vars_lines.append(", ".join(formatted_block))
        vars_html = "<br>".join(vars_lines)

        zusatz_hinweis = ""
        if is_comparing:
            zusatz_hinweis = "<br>In Klammern: Wert des Vergleichs-Runs."

        # ---- Tabelle ----
        table_html = f"""
        <div style="overflow-x: auto;">
            <table border="1" style="border-collapse: collapse; width: 100%; table-layout: fixed;">
                <colgroup>
                    <col style="width: 30%">
                    <col style="width: 20%">
                    <col style="width: 20%">
                    <col style="width: 30%">
                </colgroup>
                <tr>
                    <td style="text-align: left; vertical-align: top;">{spalte1}</td>
                    <td style="text-align: left; vertical-align: top; white-space: nowrap;">{spalte2}</td>
                    <td style="text-align: left; vertical-align: top;">{spalte3}</td>
                    <td style="text-align: left; vertical-align: top; white-space: normal; word-break: break-word;">{spalte4}</td>
                </tr>
            </table>
        </div>
        <br>
        <b>Variablen (alphabetisch absteigend):</b><br>
        {vars_html}
        <br>
        {hinweis_text}
        {zusatz_hinweis}
        """
        return widgets.HTML(table_html)

    # ====== 3. Daten laden ======
    def load_data(filename):
        nonlocal current_df, global_min_run, global_max_run
        nonlocal all_methods_full, all_prefixes_raw_full, has_none_full, category_options_full, all_filenames_full
        try:
            df = pd.read_csv(filename, delimiter=';')
        except FileNotFoundError:
            print(f"Fehler: Die Datei '{filename}' wurde nicht gefunden.")
            return None
        except Exception as e:
            print(f"Fehler beim Lesen der Datei: {e}")
            return None

        # Datumsspalte konvertieren
        if 'Zeitpunkt' in df.columns:
            df['Zeitpunkt'] = pd.to_datetime(df['Zeitpunkt'], errors='coerce')
            df = df.dropna(subset=['Zeitpunkt'])
        else:
            print("Warnung: Spalte 'Zeitpunkt' nicht gefunden.")

        # Numerische Spalten konvertieren
        numeric_cols = ['MAE', 'RMSE', 'RMSLE', 'R_Quadrat', 'Rechenzeit', 'Min_Price', 'Max_Price', 'N_Bins']
        for col in numeric_cols:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')

        # RunNo bereinigen
        if 'RunNo' not in df.columns:
            print("Fehler: Spalte 'RunNo' nicht gefunden.")
            return None
        df['RunNo'] = pd.to_numeric(df['RunNo'], errors='coerce')
        orig_len = len(df)
        df = df.dropna(subset=['RunNo'])
        if len(df) == 0:
            print("Fehler: Nach Entfernen ungültiger RunNo-Werte ist das DataFrame leer.")
            return None
        if len(df) < orig_len:
            print(f"Warnung: {orig_len - len(df)} Zeile(n) mit ungültiger RunNo wurden entfernt.")
        df['RunNo'] = df['RunNo'].astype(int)

        # Fehlende Spalten ergänzen
        if 'scriptname' not in df.columns:
            print("Warnung: Spalte scriptname nicht gefunden. 'unknown' wird gesetzt.")
            df['scriptname'] = 'unknown'
        if 'Scale_target' not in df.columns:
            print("Warnung: Spalte 'Scale_target' nicht gefunden. Verwende Standardwert False.")
            df['Scale_target'] = False

        # Präfix und Postfix aus Kommentar extrahieren
        def extract_prefix_postfix(comment):
            if pd.isna(comment):
                return None, None
            if '-' in comment:
                parts = comment.split('-', 1)
                prefix = parts[0].strip()
                postfix = parts[1].strip()
            else:
                prefix = comment.strip()
                postfix = comment.strip()
            return prefix, postfix

        df['symbol_prefix'], df['symbol_postfix'] = zip(*df['Kommentar'].apply(extract_prefix_postfix))

        # Globale min/max Runs
        global_min_run = int(df['RunNo'].min())
        global_max_run = int(df['RunNo'].max())

        # Globale Listen für Filter (vollständiger Datensatz)
        all_methods_full = sorted(df['Methode'].unique())
        all_prefixes_raw_full = sorted([p for p in df['symbol_prefix'].unique() if pd.notna(p)])
        has_none_full = df['symbol_prefix'].isna().any()
        category_options_full = [(p, p) for p in all_prefixes_raw_full]
        if has_none_full or not category_options_full:
            category_options_full.insert(0, ("(kein Präfix)", "__NONE__"))
        all_filenames_full = sorted(df['Dateiname'].unique())

        current_df = df
        return df

    # ====== 4. Hilfsfunktion zur Bestimmung des initialen DataFrames ======
    def get_initial_df_and_filters(df, run_range, startRunNo, last_n, method_filter):
        """Bestimmt den initialen DataFrame basierend auf den Parametern und extrahiert die möglichen Filterwerte."""
        if run_range is not None:
            min_run, max_run = run_range
        else:
            if startRunNo is not None:
                min_run = startRunNo
                max_run = global_max_run
            else:
                last_runs = df.sort_values('RunNo', ascending=False).head(last_n)['RunNo']
                min_run = last_runs.min()
                max_run = last_runs.max()

        initial_df = df[(df['RunNo'] >= min_run) & (df['RunNo'] <= max_run)].copy()

        if method_filter is not None:
            if isinstance(method_filter, str):
                method_filter = [method_filter]
            available_methods = initial_df['Methode'].unique()
            method_filter = [m for m in method_filter if m in available_methods]
            if method_filter:
                initial_df = initial_df[initial_df['Methode'].isin(method_filter)]

        # Extrahiere Filteroptionen aus initial_df
        init_methods = sorted(initial_df['Methode'].unique())
        init_prefixes_raw = sorted([p for p in initial_df['symbol_prefix'].unique() if pd.notna(p)])
        init_has_none = initial_df['symbol_prefix'].isna().any()
        init_category_options = [(p, p) for p in init_prefixes_raw]
        if init_has_none or not init_category_options:
            init_category_options.insert(0, ("(kein Präfix)", "__NONE__"))
        init_filenames = sorted(initial_df['Dateiname'].unique())

        return initial_df, min_run, max_run, init_methods, init_category_options, init_filenames

    # Initiales Laden
    current_df = load_data(csv_file)
    if current_df is None:
        print("Abbruch: Konnte Daten nicht laden.")
        return

    # Initialen DataFrame und Filteroptionen bestimmen
    (initial_df,
     init_min_run, init_max_run,
     init_methods, init_category_options, init_filenames) = get_initial_df_and_filters(
         current_df, run_range, startRunNo, last_n, method_filter
     )

    # ====== 5. Widgets erstellen ======
    filename_text = widgets.Text(
        value=csv_file,
        description='CSV-Datei:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='500px')
    )
    load_button = widgets.Button(
        description='Laden',
        button_style='primary',
        tooltip='Neue CSV-Datei laden'
    )

    min_run_slider = widgets.IntSlider(
        value=init_min_run,
        min=global_min_run,
        max=global_max_run,
        step=1,
        description='Min Run:',
        style={'description_width': 'initial'}
    )
    max_run_slider = widgets.IntSlider(
        value=init_max_run,
        min=global_min_run,
        max=global_max_run,
        step=1,
        description='Max Run:',
        style={'description_width': 'initial'}
    )

    # Methoden-Filter (initial mit den Werten aus initial_df)
    method_filter_widget = widgets.SelectMultiple(
        options=init_methods,
        value=method_filter if method_filter is not None else init_methods,
        description='Methode:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='300px')
    )

    # Kategorie-Filter
    category_filter_widget = widgets.SelectMultiple(
        options=init_category_options,
        value=[opt[1] for opt in init_category_options],
        description='Kategorie:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='300px')
    )

    # Dateinamen-Filter
    filename_filter_widget = widgets.SelectMultiple(
        options=init_filenames,
        value=init_filenames,
        description='Dateiname:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='400px')
    )

    # Run-Dropdowns
    run_selector_current = widgets.Dropdown(
        options=[],
        value=None,
        description='Aktueller Run:',
        style={'description_width': 'initial'}
    )
    run_selector_compare = widgets.Dropdown(
        options=[],
        value=None,
        description='Vergleichs-Run:',
        style={'description_width': 'initial'}
    )

    # Output-Bereiche
    span_out = widgets.Output()
    plot_out = widgets.Output()
    offset_hinweis_out = widgets.Output()
    out = widgets.Output()
    current_df_container = [initial_df]   # Container für den jeweils gefilterten DataFrame

    # ====== 6. Update-Funktionen ======
    def update_table(change=None):
        current_no = run_selector_current.value
        compare_no = run_selector_compare.value
        df_current = current_df_container[0]
        if df_current is None or current_no is None:
            return
        current_row = df_current[df_current['RunNo'] == current_no].iloc[0]
        if compare_no != current_no and compare_no is not None:
            compare_row = df_current[df_current['RunNo'] == compare_no].iloc[0]
        else:
            compare_row = None
        with out:
            clear_output()
            display(create_info_table(current_row, compare_row))

    def update_range(change=None):
        min_run = min_run_slider.value
        max_run = max_run_slider.value
        if min_run > max_run:
            min_run, max_run = max_run, min_run
            min_run_slider.value = min_run
            max_run_slider.value = max_run
            return

        # Basis-Filter nach Run-Nummer
        df_filtered = current_df[
            (current_df['RunNo'] >= min_run) & (current_df['RunNo'] <= max_run)
        ].copy()

        # Extrahiere die aktuell möglichen Werte für die Filter aus df_filtered
        current_methods = sorted(df_filtered['Methode'].unique())
        current_prefixes_raw = sorted([p for p in df_filtered['symbol_prefix'].unique() if pd.notna(p)])
        current_has_none = df_filtered['symbol_prefix'].isna().any()
        current_category_options = [(p, p) for p in current_prefixes_raw]
        if current_has_none or not current_category_options:
            current_category_options.insert(0, ("(kein Präfix)", "__NONE__"))
        current_filenames = sorted(df_filtered['Dateiname'].unique())

        # Temporär die Observer der Filter-Widgets deaktivieren, um Rekursion zu vermeiden
        method_filter_widget.unobserve(update_range, names='value')
        category_filter_widget.unobserve(update_range, names='value')
        filename_filter_widget.unobserve(update_range, names='value')

        # Widget-Optionen auf die aktuell möglichen Werte setzen
        method_filter_widget.options = current_methods
        # Wert auf die Schnittmenge der aktuell ausgewählten Methoden mit den neuen Optionen setzen
        old_methods = method_filter_widget.value
        new_methods = [m for m in old_methods if m in current_methods]
        if not new_methods and current_methods:
            new_methods = [current_methods[0]]
        method_filter_widget.value = new_methods

        category_filter_widget.options = current_category_options
        old_cats = category_filter_widget.value
        new_cats = [c for c in old_cats if c in [opt[1] for opt in current_category_options]]
        if not new_cats and current_category_options:
            new_cats = [current_category_options[0][1]]
        category_filter_widget.value = new_cats

        filename_filter_widget.options = current_filenames
        old_files = filename_filter_widget.value
        new_files = [f for f in old_files if f in current_filenames]
        if not new_files and current_filenames:
            new_files = [current_filenames[0]]
        filename_filter_widget.value = new_files

        # Observer wieder aktivieren
        method_filter_widget.observe(update_range, names='value')
        category_filter_widget.observe(update_range, names='value')
        filename_filter_widget.observe(update_range, names='value')

        # Jetzt die Filter (Methode, Kategorie, Dateiname) anwenden
        selected_methods = method_filter_widget.value
        if selected_methods and len(selected_methods) < len(current_methods):
            df_filtered = df_filtered[df_filtered['Methode'].isin(selected_methods)]

        selected_cats = category_filter_widget.value
        if selected_cats and len(selected_cats) < len(current_category_options):
            cond = pd.Series(False, index=df_filtered.index)
            for cat_val in selected_cats:
                if cat_val == "__NONE__":
                    cond |= df_filtered['symbol_prefix'].isna()
                else:
                    cond |= (df_filtered['symbol_prefix'] == cat_val)
            df_filtered = df_filtered[cond]

        selected_files = filename_filter_widget.value
        if selected_files and len(selected_files) < len(current_filenames):
            df_filtered = df_filtered[df_filtered['Dateiname'].isin(selected_files)]

        if df_filtered.empty:
            with plot_out:
                clear_output()
                display(widgets.HTML("<b>Keine Daten im gewählten Bereich mit den aktuellen Filtern.</b>"))
            run_selector_current.options = []
            run_selector_compare.options = []
            current_df_container[0] = None
            with span_out:
                clear_output()
            with offset_hinweis_out:
                clear_output()
            return

        current_df_container[0] = df_filtered

        # Spannweiten-Text
        min_time = df_filtered['Zeitpunkt'].min()
        max_time = df_filtered['Zeitpunkt'].max()
        min_run = df_filtered['RunNo'].min()
        max_run = df_filtered['RunNo'].max()
        min_rmse = df_filtered['RMSE'].min()
        max_rmse = df_filtered['RMSE'].max()
        min_rmsle = df_filtered['RMSLE'].min()
        max_rmsle = df_filtered['RMSLE'].max()
        min_mae = df_filtered['MAE'].min()
        max_mae = df_filtered['MAE'].max()
        min_r2 = df_filtered['R_Quadrat'].min()
        max_r2 = df_filtered['R_Quadrat'].max()
        min_rechenzeit = df_filtered['Rechenzeit'].min()
        max_rechenzeit = df_filtered['Rechenzeit'].max()

        span_text = (f"Zeitpunkt: {min_time.strftime('%d.%m.%Y %H:%M:%S')} bis {max_time.strftime('%d.%m.%Y %H:%M:%S')} | "
                     f"RunNo: {min_run} bis {max_run} | "
                     f"RMSE: {min_rmse:.2f} bis {max_rmse:.2f} | RMSLE: {min_rmsle:.4f} bis {max_rmsle:.4f} | "
                     f"MAE: {min_mae:.2f} bis {max_mae:.2f} | R²: {min_r2:.4f} bis {max_r2:.4f} | "
                     f"Rechenzeit: {min_rechenzeit:.2f}s bis {max_rechenzeit:.2f}s")

        with span_out:
            clear_output()
            display(widgets.HTML(f"<b>Spannweiten (dargestellte Daten):</b> {span_text}"))

        fig, offset_applied = create_figure(df_filtered)
        with plot_out:
            clear_output()
            display(fig)

        run_options = [(f"Run {r}", r) for r in sorted(df_filtered['RunNo'].unique())]
        run_selector_current.options = run_options
        run_selector_compare.options = run_options

        if run_selector_current.value not in df_filtered['RunNo'].values:
            run_selector_current.value = df_filtered['RunNo'].max()
        if run_selector_compare.value not in df_filtered['RunNo'].values:
            if len(df_filtered) >= 2:
                run_selector_compare.value = df_filtered['RunNo'].nlargest(2).iloc[-1]
            else:
                run_selector_compare.value = run_selector_current.value

        with offset_hinweis_out:
            clear_output()
            if offset_applied:
                display(Markdown("**Hinweis:** Punkte mit identischen Koordinaten wurden minimal versetzt, um Überlagerungen sichtbar zu machen."))

        update_table()

    def on_load_button_clicked(b):
        new_file = filename_text.value.strip()
        new_df = load_data(new_file)
        if new_df is None:
            return

        # Nach dem Laden den initialen Bereich und die Filteroptionen neu bestimmen
        (new_initial_df,
         new_min_run, new_max_run,
         new_init_methods, new_init_category_options, new_init_filenames) = get_initial_df_and_filters(
             new_df, run_range, startRunNo, last_n, method_filter
         )

        # Slider-Bereiche anpassen
        min_run_slider.min = global_min_run
        min_run_slider.max = global_max_run
        max_run_slider.min = global_min_run
        max_run_slider.max = global_max_run
        min_run_slider.value = new_min_run
        max_run_slider.value = new_max_run

        # Filter-Widgets auf die neuen Optionen setzen
        method_filter_widget.unobserve(update_range, names='value')
        category_filter_widget.unobserve(update_range, names='value')
        filename_filter_widget.unobserve(update_range, names='value')

        method_filter_widget.options = new_init_methods
        method_filter_widget.value = method_filter if method_filter is not None else new_init_methods

        category_filter_widget.options = new_init_category_options
        category_filter_widget.value = [opt[1] for opt in new_init_category_options]

        filename_filter_widget.options = new_init_filenames
        filename_filter_widget.value = new_init_filenames

        method_filter_widget.observe(update_range, names='value')
        category_filter_widget.observe(update_range, names='value')
        filename_filter_widget.observe(update_range, names='value')

        # update_range wird durch die Slider-Änderung automatisch aufgerufen,
        # aber wir rufen es sicherheitshalber auch direkt auf, um den Container zu aktualisieren.
        update_range()

    load_button.on_click(on_load_button_clicked)

    # Observer
    min_run_slider.observe(update_range, names='value')
    max_run_slider.observe(update_range, names='value')
    method_filter_widget.observe(update_range, names='value')
    category_filter_widget.observe(update_range, names='value')
    filename_filter_widget.observe(update_range, names='value')
    run_selector_current.observe(update_table, names='value')
    run_selector_compare.observe(update_table, names='value')

    # ====== 7. Layout ======
    display(widgets.HBox([filename_text, load_button]))
    display(span_out)
    display(plot_out)
    display(offset_hinweis_out)
    display(widgets.HBox([min_run_slider, max_run_slider]))
    display(widgets.HBox([
        method_filter_widget,
        category_filter_widget,
        filename_filter_widget
    ]))
    display(widgets.HBox([run_selector_current, run_selector_compare]))
    display(out)

    # Initial zeichnen
    update_range()

# Beispielaufruf
plot_results_dropdown(last_n=500, csv_file='Ames_Regression_interaktiv_log.csv')

Output()

Output()

Output()

Output()

In [2]:

# ====== Beispielaufrufe (auskommentiert) ======
# Alle letzten 10 Runs:

#plot_results_dropdown(last_n=50, csv_file='Ames_Regression_interaktiv_log.csv')

# Nur bestimmte Methoden:
plot_results_dropdown( startRunNo =90)

# Bestimmter Run-Bereich:
#plot_results_dropdown(run_range=(90,120))

Output()

Output()

Output()

Output()